# Resolution-matched edge features: different blur for the photo and the map (iteration 31)

**Why.** Iteration 30 (figure and scale study): at every height the finest gradient scale (sigma 1 px) is best and coarser scales are worse, so larger structure does NOT take over. In the figure the map at the true place is crisp (0.3 m satellite resampled) while the thermal photo is smoother and low contrast, so the same sigma on both sides compares an image with a blurrier one. Try a different gradient blur for the canvas and for the map window: sigma_canvas in {0.7, 1, 1.5, 2}, sigma_map in {0.7, 1, 1.5, 2, 3}. (The map is more detailed than the photo, so the map may need MORE blur to look like the photo.)

**Kill rule (written before the run).** On development photos above 800 m (142) a pair is useful only if the share with the best peak within 2 sigma of the truth is clearly above the (1, 1) baseline AND wrong (beyond 8 sigma) is not higher than the baseline. A gain only on one height band, or a gain that needs a different pair per band, does not count. Development photos only; one query (heading 0, scale 1.0, hint jittered 5 km).

In [1]:
import os, sys, math, colorsys
import numpy as np, pandas as pd, torch, cv2
HEADLESS = os.environ.get("EF_HEADLESS", "../../../../Main_Pipeline_Headless"); sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [int(v) for v in os.environ.get("EF_LEVELS", "3,4,5,6,7,8,9,10,11").split(",")]
from geoloc import photo, queries, scoring
from geoloc.cli import read_rows, move_hint
from geoloc.coarse_search import DEVICE, score_map_for_query, pick_peaks, window_transforms, zncc_everywhere, fast_length, cut_map_features
from geoloc.edge_features import normalise_cells
from geoloc.levels import level_for_height, level_gsd, map_piece
from geoloc.map_store import MapStore
CELL = settings.CELL; TILE = settings.TILE
DATASET = os.environ["EF_DATASET"]; STORE = os.environ["EF_STORE"]; OUT = os.environ.get("EF_OUT", "results/edge_features"); os.makedirs(OUT, exist_ok=True)
store = MapStore(STORE); rows = read_rows(DATASET)       # the dataset folder holds the development photos only
heights = np.array([float(r["alt"]) for r in rows]); print(len(rows), "photos, heights", heights.min(), heights.max())

332 photos, heights 100.599998 2016.599976


In [2]:
def edge_cells(image, sigma, cell_valid=None):
    # 8-direction edge strength summed over 4x4-pixel cells, gradient blur = sigma pixels (sigma 1 = the pipeline)
    blurred = cv2.GaussianBlur(image.astype(np.float32), (0, 0), sigma)
    gx = np.zeros_like(blurred); gy = np.zeros_like(blurred)
    gx[:, 1:-1] = blurred[:, 2:] - blurred[:, :-2]; gy[1:-1, :] = blurred[2:, :] - blurred[:-2, :]
    strength = np.hypot(gx, gy); angle = np.arctan2(gy, gx) % np.pi; bin_id = np.minimum((angle / np.pi * settings.N_BINS).astype(int), settings.N_BINS - 1)
    r, c = image.shape[0] // CELL, image.shape[1] // CELL; out = np.zeros((settings.N_BINS, r, c), np.float32)
    for b in range(settings.N_BINS):
        out[b] = (strength * (bin_id == b))[:r * CELL, :c * CELL].reshape(r, CELL, c, CELL).sum(axis=(1, 3))
    return out

def canvas_features(canvas, valid, sigma):
    safe = cv2.erode(valid.astype(np.uint8), np.ones((7, 7), np.uint8)); side = canvas.shape[0] // CELL
    cell_valid = safe.reshape(side, CELL, side, CELL).mean(axis=(1, 3)) >= 0.8
    return normalise_cells(edge_cells(canvas, sigma), cell_valid) * cell_valid, cell_valid

def window_pixels(level_image, top, left, cells_wide):
    size = cells_wide * CELL; r0, c0 = top * CELL, left * CELL; piece = np.zeros((size, size), np.float32); h, w = level_image.shape
    a0, a1 = max(r0, 0), min(r0 + size, h); b0, b1 = max(c0, 0), min(c0 + size, w)
    if a1 > a0 and b1 > b0: piece[a0 - r0:a1 - r0, b0 - c0:b1 - c0] = np.asarray(level_image[a0:a1, b0:b1], np.float32) / 255.0
    return piece

def map_features(piece, sigma):
    feats = edge_cells(piece, sigma); has = (piece[:feats.shape[1] * CELL, :feats.shape[2] * CELL].reshape(feats.shape[1], CELL, feats.shape[2], CELL).min(axis=(1, 3)) > 0)
    has = cv2.erode(has.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    feats = normalise_cells(feats, has) * has; return feats, has

def hsv_picture(feats):
    # hue = dominant edge direction, brightness = total strength (99th percentile = full)
    total = feats.sum(axis=0); hue = feats.argmax(axis=0) / settings.N_BINS; val = np.clip(total / (np.percentile(total, 99) + 1e-6), 0, 1)
    rgb = np.zeros(total.shape + (3,), np.float32)
    for i in range(total.shape[0]):
        for j in range(total.shape[1]): rgb[i, j] = colorsys.hsv_to_rgb(hue[i, j], 0.9, val[i, j])
    return rgb

In [3]:
CANVAS_SIGMAS = [0.7, 1.0, 1.5, 2.0]; MAP_SIGMAS = [0.7, 1.0, 1.5, 2.0, 3.0]; records = []; query = queries.query_list()[0]
for index, meta in enumerate(rows):
    if float(meta["alt"]) < 800: continue
    meta = move_hint(meta, index, 5000.0, 0); camera = photo.read_camera(meta); pose = photo.read_pose(meta)
    prior = store.gps_to_metres(float(meta["lat_sc"]), float(meta["lon_sc"])); truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"]))
    sigma_m = scoring.sigma_m(pose["height"], camera["fx"]); level = level_for_height(pose["height"], camera); gsd = level_gsd(level)
    picture = photo.prepare_photo(os.path.join(DATASET, "images", meta["filename"]), pose, camera, level)
    canvas, valid = photo.draw_canvas(picture, pose, camera, level, query["heading"], query["scale_factor"]); side = TILE // CELL
    reach = settings.DISC_RADIUS / gsd + side * CELL / 2; cells_wide = int(math.ceil(2 * reach / CELL))
    top = int(round((prior[1] / gsd - reach) / CELL)); left = int(round((prior[0] / gsd - reach) / CELL))
    piece = window_pixels(store.levels[level], top, left, cells_wide); size = (fast_length(cells_wide), fast_length(cells_wide))
    ids = np.arange(cells_wide - side + 1); y_m = ((top + ids) * CELL + side * CELL / 2) * gsd; x_m = ((left + ids) * CELL + side * CELL / 2) * gsd
    inside = (x_m[None, :] - prior[0]) ** 2 + (y_m[:, None] - prior[1]) ** 2 <= settings.DISC_RADIUS ** 2
    for sm in MAP_SIGMAS:
        mf, mh = map_features(piece, sm); window = torch.from_numpy(mf).to(DEVICE); shared = window_transforms(window, size)
        for sc in CANVAS_SIGMAS:
            kernel, cells = canvas_features(canvas, valid, sc)
            zn = zncc_everywhere(window, shared, size, torch.from_numpy(kernel).to(DEVICE), torch.from_numpy(cells.astype(np.float32)).to(DEVICE)).cpu().numpy()
            usable = (zn > -1) & inside; v = zn[usable]
            if usable.sum() < 100: continue
            flat = np.where(usable, zn, -9); best = np.unravel_index(np.argmax(flat), flat.shape)
            records.append(dict(photo=index, height=pose["height"], sigma_canvas=sc, sigma_map=sm, err=math.hypot(x_m[best[1]] - truth[0], y_m[best[0]] - truth[1]) / sigma_m,
                                z_best=(flat[best] - v.mean()) / v.std()))
    if index % 20 == 0: print(index, flush=True)
grid = pd.DataFrame(records); grid.to_csv(os.path.join(OUT, "sigma_grid.csv"), index=False)
grid["band"] = pd.cut(grid.height, [800, 1100, 1400, 1700, 2100])
agg = grid.groupby(["sigma_canvas", "sigma_map"]).agg(n=("err", "size"), right2=("err", lambda e: (e <= 2).mean()), within1=("err", lambda e: (e <= 1).mean()), wrong=("err", lambda e: (e > 8).mean()), z_best=("z_best", "median")).round(3)
print("all photos above 800 m (rows = canvas sigma, columns = map sigma)")
for col in ["right2", "wrong", "within1"]: print(col); print(agg[col].unstack("sigma_map").to_string())
print("\nright2 by height band for the baseline (1,1) and the best pair per band")
by = grid.groupby(["band", "sigma_canvas", "sigma_map"], observed=True).err.agg(lambda e: (e <= 2).mean()).round(2)
for band, g in by.groupby(level=0): print(band, "baseline", g.loc[(band, 1.0, 1.0)], "best", g.idxmax()[1:], g.max())

20


40


60


160


200


220


... (2 outputs left out)

280


300


320


all photos above 800 m (rows = canvas sigma, columns = map sigma)
right2
sigma_map       0.7    1.0    1.5    2.0    3.0
sigma_canvas                                   
0.7           0.599  0.606  0.585  0.521  0.373
1.0           0.627  0.606  0.570  0.521  0.401
1.5           0.493  0.528  0.521  0.486  0.394
2.0           0.387  0.437  0.451  0.423  0.380
wrong
sigma_map       0.7    1.0    1.5    2.0    3.0
sigma_canvas                                   
0.7           0.190  0.176  0.232  0.310  0.493
1.0           0.197  0.197  0.246  0.324  0.486
1.5           0.324  0.282  0.289  0.331  0.472
2.0           0.458  0.415  0.380  0.401  0.507
within1
sigma_map       0.7    1.0    1.5    2.0    3.0
sigma_canvas                                   
0.7           0.218  0.246  0.239  0.211  0.113
1.0           0.239  0.239  0.218  0.197  0.127
1.5           0.183  0.204  0.197  0.176  0.127
2.0           0.148  0.162  0.148  0.141  0.134

right2 by height band for the baseline (1,1) and